# Notebook 2: el analista responde las preguntas

**Obligatorio Herramientas de Software para Big Data, Parte 1. Vuelos regulares de Brasil, 2025.**
Grupo: Pilar Fernández, Jerónimo Brum y Fernando Tonarelli.

Este notebook hace de analista: arranca de las tablas de Hive que dejó el notebook 1 (base `vuelos_brasil`) y responde las cinco preguntas. Todo sale de consultas `spark.sql` sobre esas tablas; no se lee ningún archivo de HDFS directo.

Las preguntas, tal como se aprobaron:

1. ¿Qué aerolínea mejoró más su tasa de puntualidad a lo largo del año (vuelos con más de 15 minutos de atraso), separando vuelos nacionales e internacionales y considerando solo empresas con al menos 100 vuelos por mes?
2. ¿Qué rutas con al menos 100 vuelos programados tienen mayor tasa de cancelación, y esas cancelaciones se concentran en una o dos aerolíneas?
3. En los 10 aeropuertos con más movimiento, ¿cómo cambia el atraso promedio de salida según la hora del día y el día de la semana?
4. ¿Qué países de destino ganaron o perdieron frecuencias semanales con Brasil, Uruguay incluido, comparando el primer y el último trimestre del año, y cómo cambió la participación de empresas brasileñas frente a extranjeras en esas rutas?
5. ¿Los aeropuertos grandes, medianos y chicos tienen distinta tasa de puntualidad y de cancelación, y cómo evoluciona esa diferencia a lo largo del año?

| Paso | Qué hace | Salida |
|---|---|---|
| 0 | Configuración | Sesión de Spark con Hive y tablas del modelo |
| 1 | Definiciones comunes | Qué es un vuelo programado, medido, puntual y cancelado |
| 2 a 4 | P1, P2 y P3 | Consulta, resultado en pandas y dos gráficos por pregunta |
| 5 | P4 y P5 | Tablas externas en `/anl`, base `resumen_vuelos_brasil`, para Superset |
| 6 | Validación | Tablas, tipo, ubicación y DDL |

## 0. Configuración

Antes de correr, en la terminal de la VM:

```
docker exec namenode hdfs dfs -mkdir -p /anl/vuelos_brasil/v1
docker exec namenode hdfs dfs -chmod -R 777 /anl
```

Es lo mismo que con `/rfn` en el notebook 1: Hive y Spark escriben con usuarios distintos.

In [ ]:
from pyspark.sql import SparkSession
from matplotlib.ticker import PercentFormatter
import matplotlib.pyplot as plt
import seaborn as sns

spark = (SparkSession.builder
         .appName("nb2_vuelos_brasil_analisis")
         .enableHiveSupport()
         .getOrCreate())

VERSION = "v1"                      # misma versión que el modelo del notebook 1
PREFIX = ""                         # vacío = HDFS por defecto de la VM
DB = "vuelos_brasil"                # modelo que dejó el notebook 1
DB_ANL = "resumen_vuelos_brasil"    # respuestas de P4 y P5, la base que usa Superset
ANL = f"{PREFIX}/anl/vuelos_brasil/{VERSION}"

# Colores en orden fijo, distinguibles con daltonismo; el azul y el rojo marcan mejora y empeoramiento
PALETA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
GRIS = "#c3c2b7"
sns.set_theme(style="whitegrid", palette=PALETA)
spark.sql(f"SHOW TABLES IN {DB}").show()

## 1. Definiciones que usan todas las preguntas

Todas salen de columnas que el notebook 1 dejó en `fact_vuelo`, donde solo hay vuelos de pasajeros (N e I).

| Término | Definición | Columnas |
|---|---|---|
| Vuelo programado | Tiene horario previsto y su situación es realizado o cancelado. Los no programados (código DI distinto de 0) no tienen horario previsto y quedan afuera | `tiene_horario_previsto`, `informado` |
| Tasa de cancelación | Cancelados sobre programados | `cancelado` |
| Vuelo medido | Realizado y con atraso de salida calculado. No entran los no programados ni los de atraso extremo, que son errores de registro | `puntual` no nulo |
| Tasa de puntualidad | Vuelos medidos que salieron con 15 minutos de atraso o menos, sobre vuelos medidos. Es el complemento de la proporción de vuelos con más de 15 minutos de atraso, que es como lo plantea P1 | `puntual` |
| Atraso de salida | Hora real menos hora prevista, en minutos. Las salidas adelantadas (hasta 60 minutos) entran con signo negativo | `atraso_partida_min` |

En SQL, `COUNT(columna)` no cuenta los nulos y `CAST(true AS INT)` vale 1. Por eso `SUM(CAST(puntual AS INT)) / COUNT(puntual)` es la tasa de puntualidad sin filtrar a mano los vuelos que no se pueden medir.

## 2. P1: qué aerolínea mejoró más su puntualidad

Cómo se mide:

- **Mejora**: puntualidad del cuarto trimestre menos la del primero, en puntos porcentuales. Dos trimestres son más estables que dos meses sueltos.
- La puntualidad de un trimestre se calcula con todos sus vuelos medidos juntos, no como promedio de los tres meses.
- Nacionales (`tipo_linha` N) e internacionales (I) van por separado. Una empresa puede entrar en un segmento y no en el otro.
- Entran las empresas con 100 vuelos medidos o más **en cada uno de los 12 meses**, dentro del segmento.

In [ ]:
p1_mensual = spark.sql(f"""
WITH mensual AS (
    SELECT f.tipo_linha, f.icao_empresa, f.anio_mes, d.trimestre,
           COUNT(f.puntual)            AS vuelos_medidos,
           SUM(CAST(f.puntual AS INT)) AS vuelos_puntuales
    FROM {DB}.fact_vuelo f
    JOIN {DB}.dim_fecha d ON f.fecha = d.fecha
    GROUP BY f.tipo_linha, f.icao_empresa, f.anio_mes, d.trimestre
),
elegibles AS (
    SELECT tipo_linha, icao_empresa
    FROM mensual
    GROUP BY tipo_linha, icao_empresa
    HAVING COUNT(*) = 12 AND MIN(vuelos_medidos) >= 100
)
SELECT m.tipo_linha, m.icao_empresa, e.nombre_empresa, e.nacionalidad, m.anio_mes, m.trimestre,
       m.vuelos_medidos, m.vuelos_puntuales,
       m.vuelos_puntuales / m.vuelos_medidos AS tasa_puntualidad
FROM mensual m
JOIN elegibles g ON m.tipo_linha = g.tipo_linha AND m.icao_empresa = g.icao_empresa
JOIN {DB}.dim_empresa e ON m.icao_empresa = e.icao_empresa
""")
p1_mensual.createOrReplaceTempView("p1_mensual")
spark.sql("""
SELECT tipo_linha, COUNT(DISTINCT icao_empresa) AS empresas_que_entran
FROM p1_mensual
GROUP BY tipo_linha
""").show()

In [ ]:
p1_mejora = spark.sql("""
SELECT tipo_linha, icao_empresa, nombre_empresa, nacionalidad,
       ROUND(puntualidad_t1, 4) AS puntualidad_t1,
       ROUND(puntualidad_t4, 4) AS puntualidad_t4,
       ROUND((puntualidad_t4 - puntualidad_t1) * 100, 1) AS mejora_pp
FROM (
    SELECT tipo_linha, icao_empresa, nombre_empresa, nacionalidad,
           SUM(IF(trimestre = 1, vuelos_puntuales, 0)) / SUM(IF(trimestre = 1, vuelos_medidos, 0)) AS puntualidad_t1,
           SUM(IF(trimestre = 4, vuelos_puntuales, 0)) / SUM(IF(trimestre = 4, vuelos_medidos, 0)) AS puntualidad_t4
    FROM p1_mensual
    GROUP BY tipo_linha, icao_empresa, nombre_empresa, nacionalidad
) t
ORDER BY tipo_linha, mejora_pp DESC
""")
p1_mejora.show(40, truncate=30)

In [ ]:
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "set", "oct", "nov", "dic"]
SEGMENTOS = {"N": "Nacionales", "I": "Internacionales"}

def etiqueta(df):
    # código ICAO y nombre acortado, para ejes y leyendas
    return df["icao_empresa"] + " " + df["nombre_empresa"].fillna("").str.title().str.slice(0, 22)

mensual, mejora = p1_mensual.toPandas(), p1_mejora.toPandas()
mensual["empresa"], mejora["empresa"] = etiqueta(mensual), etiqueta(mejora)
mensual["mes"] = mensual["anio_mes"].str[5:].astype(int)
top5 = mejora.sort_values("mejora_pp", ascending=False).groupby("tipo_linha").head(5)

# Gráfico 1: puntualidad mes a mes de las cinco que más mejoraron en cada segmento
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, (tipo, nombre) in zip(axes, SEGMENTOS.items()):
    elegidas = top5.loc[top5["tipo_linha"] == tipo, "icao_empresa"]
    datos = mensual[(mensual["tipo_linha"] == tipo) & mensual["icao_empresa"].isin(elegidas)]
    sns.lineplot(data=datos, x="mes", y="tasa_puntualidad", hue="empresa", marker="o", ax=ax)
    ax.set_xticks(range(1, 13), MESES)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set_title(f"{nombre}: las que más mejoraron")
    ax.set_xlabel("Mes de 2025")
    ax.set_ylabel("Vuelos con 15 min de atraso o menos")
    ax.legend(title=None, fontsize=8)
fig.suptitle("P1. Puntualidad mensual por aerolínea")
plt.tight_layout()
plt.show()

# Gráfico 2: mejora de todas las empresas que entran, en puntos porcentuales
cantidad = mejora["tipo_linha"].value_counts()
fig, axes = plt.subplots(2, 1, figsize=(10, 2 + 0.35 * len(mejora)),
                         gridspec_kw={"height_ratios": [cantidad.get("N", 1), cantidad.get("I", 1)]})
for ax, (tipo, nombre) in zip(axes, SEGMENTOS.items()):
    datos = mejora[mejora["tipo_linha"] == tipo].sort_values("mejora_pp", ascending=False)
    ax.barh(datos["empresa"], datos["mejora_pp"],
            color=[PALETA[0] if v >= 0 else PALETA[7] for v in datos["mejora_pp"]])
    ax.invert_yaxis()
    ax.axvline(0, color="grey", linewidth=0.8)
    ax.set_title(f"{nombre} ({len(datos)} empresas)")
    ax.set_xlabel("Puntualidad del cuarto trimestre menos la del primero (puntos porcentuales)")
fig.suptitle("P1. Cambio de puntualidad entre el primer y el último trimestre")
plt.tight_layout()
plt.show()

**Control antes de concluir.** Si una empresa cambia la forma de anotar la hora real de salida, su puntualidad salta sin que la operación haya cambiado. En estos datos es normal que muchos vuelos salgan unos minutos antes de lo previsto, y esa proporción se mueve poco de un mes a otro. Por eso, para las tres que más mejoraron en cada segmento, se mira el porcentaje de salidas adelantadas mes a mes: un salto brusco de un mes al siguiente apunta a un cambio en el registro más que a una mejora real.

In [ ]:
top3 = mejora.sort_values("mejora_pp", ascending=False).groupby("tipo_linha").head(3)
claves = ", ".join(f"'{t}{e}'" for t, e in zip(top3["tipo_linha"], top3["icao_empresa"]))
adelantadas = spark.sql(f"""
SELECT tipo_linha, icao_empresa, anio_mes,
       ROUND(AVG(CAST(atraso_partida_min < 0 AS INT)) * 100) AS pct_salidas_adelantadas
FROM {DB}.fact_vuelo
WHERE puntual IS NOT NULL AND CONCAT(tipo_linha, icao_empresa) IN ({claves})
GROUP BY tipo_linha, icao_empresa, anio_mes
""").toPandas()
adelantadas.pivot_table(index=["tipo_linha", "icao_empresa"], columns="anio_mes",
                        values="pct_salidas_adelantadas")

**Respuesta P1** (completar con la salida al correrlo en la VM): qué empresa mejoró más en cada segmento y cuántos puntos, y si el control de salidas adelantadas confirma que la mejora es real.

## 3. P2: rutas con mayor tasa de cancelación

Cómo se mide:

- **Ruta**: el par de aeropuertos, sin importar el sentido. Congonhas a Santos Dumont y la vuelta son la misma ruta, porque una cancelación casi siempre afecta a la ida y a la vuelta.
- **Tasa de cancelación**: cancelados sobre programados en el año. Entran las rutas con 100 programados o más, sumando los dos sentidos.
- **Concentración**: qué parte de las cancelaciones de cada ruta aporta la primera aerolínea y qué parte las dos primeras.
- **Solo rutas que tocan Brasil.** En los datos hay vuelos de pasajeros entre dos aeropuertos extranjeros, y varias de esas rutas figuran canceladas todo el año (por ejemplo Buenos Aires a Madrid). No son rutas de Brasil y llenarían el ranking con un 100% de cancelación. La primera celda los cuenta.

In [ ]:
# Vuelos programados con su ruta (par de aeropuertos en orden alfabético) y si tocan Brasil
spark.sql(f"""
SELECT f.icao_empresa, f.cancelado,
       LEAST(f.icao_origem, f.icao_destino)    AS aeropuerto_a,
       GREATEST(f.icao_origem, f.icao_destino) AS aeropuerto_b,
       COALESCE(o.iso_pais = 'BR' OR d.iso_pais = 'BR', false) AS toca_brasil
FROM {DB}.fact_vuelo f
JOIN {DB}.dim_aeropuerto o ON f.icao_origem = o.icao_aeropuerto
JOIN {DB}.dim_aeropuerto d ON f.icao_destino = d.icao_aeropuerto
WHERE f.tiene_horario_previsto AND f.informado
""").createOrReplaceTempView("p2_programados")

spark.sql("""
SELECT toca_brasil, COUNT(*) AS programados, SUM(CAST(cancelado AS INT)) AS cancelados,
       ROUND(SUM(CAST(cancelado AS INT)) / COUNT(*), 3) AS tasa_cancelacion
FROM p2_programados
GROUP BY toca_brasil
""").show()

In [ ]:
p2_rutas = spark.sql(f"""
SELECT CONCAT(COALESCE(a.ciudad, r.aeropuerto_a), ' - ', COALESCE(b.ciudad, r.aeropuerto_b),
              ' (', r.aeropuerto_a, '-', r.aeropuerto_b, ')') AS ruta,
       r.aeropuerto_a, r.aeropuerto_b, r.aerolineas, r.programados, r.cancelados,
       ROUND(r.cancelados / r.programados, 4) AS tasa_cancelacion
FROM (
    SELECT aeropuerto_a, aeropuerto_b, COUNT(DISTINCT icao_empresa) AS aerolineas,
           COUNT(*) AS programados, SUM(CAST(cancelado AS INT)) AS cancelados
    FROM p2_programados
    WHERE toca_brasil
    GROUP BY aeropuerto_a, aeropuerto_b
    HAVING COUNT(*) >= 100
) r
JOIN {DB}.dim_aeropuerto a ON r.aeropuerto_a = a.icao_aeropuerto
JOIN {DB}.dim_aeropuerto b ON r.aeropuerto_b = b.icao_aeropuerto
ORDER BY tasa_cancelacion DESC, r.programados DESC, ruta
LIMIT 10
""")
p2_rutas.createOrReplaceTempView("p2_top")
p2_rutas.show(truncate=60)

In [ ]:
# Cancelaciones de cada ruta del top por aerolínea, con su puesto dentro de la ruta
spark.sql(f"""
SELECT ruta, tasa_cancelacion, icao_empresa, nombre_empresa, cancelados, participacion,
       ROW_NUMBER() OVER (PARTITION BY ruta ORDER BY cancelados DESC, icao_empresa) AS puesto
FROM (
    SELECT t.ruta, t.tasa_cancelacion, p.icao_empresa, e.nombre_empresa,
           COUNT(*) AS cancelados, COUNT(*) / t.cancelados AS participacion
    FROM p2_programados p
    JOIN p2_top t ON p.aeropuerto_a = t.aeropuerto_a AND p.aeropuerto_b = t.aeropuerto_b
    JOIN {DB}.dim_empresa e ON p.icao_empresa = e.icao_empresa
    WHERE p.cancelado
    GROUP BY t.ruta, t.tasa_cancelacion, t.cancelados, p.icao_empresa, e.nombre_empresa
) x
""").createOrReplaceTempView("p2_empresas")

spark.sql("""
SELECT ruta,
       MAX(IF(puesto = 1, icao_empresa, NULL))          AS primera,
       ROUND(MAX(IF(puesto = 1, participacion, 0)), 3)  AS part_primera,
       MAX(IF(puesto = 2, icao_empresa, NULL))          AS segunda,
       ROUND(SUM(IF(puesto <= 2, participacion, 0)), 3) AS part_dos_primeras
FROM p2_empresas
GROUP BY ruta
ORDER BY MAX(tasa_cancelacion) DESC
""").show(truncate=60)

In [ ]:
rutas = p2_rutas.toPandas()
empresas = spark.table("p2_empresas").toPandas()
empresas["empresa"] = etiqueta(empresas)

# Gráfico 1: tasa de cancelación de las 10 rutas
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=rutas, x="tasa_cancelacion", y="ruta", color=PALETA[0], ax=ax)
for i, fila in rutas.iterrows():
    ax.text(fila["tasa_cancelacion"], i, f"  {fila['cancelados']} de {fila['programados']}",
            va="center", fontsize=8)
ax.set_xlim(0, rutas["tasa_cancelacion"].max() * 1.2)   # lugar para las etiquetas
ax.set_xticks([i / 5 for i in range(6)])
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set_title("P2. Las 10 rutas con mayor tasa de cancelación (100 vuelos programados o más)")
ax.set_xlabel("Cancelados sobre programados, 2025")
ax.set_ylabel(None)
plt.tight_layout()
plt.show()

# Gráfico 2: qué aerolínea aporta las cancelaciones de cada ruta
# Pasadas las 7 aerolíneas con más cancelaciones, el resto va junto en "Otras" (gris) para no repetir colores
principales = list(empresas.groupby("empresa")["cancelados"].sum().nlargest(7).index)
empresas["grupo"] = empresas["empresa"].where(empresas["empresa"].isin(principales), "Otras")
participacion = (empresas.pivot_table(index="ruta", columns="grupo", values="participacion", aggfunc="sum")
                 .reindex(index=rutas["ruta"], columns=principales + ["Otras"]).fillna(0))
participacion = participacion.loc[:, participacion.sum() > 0]
colores = [dict(zip(principales, PALETA)).get(c, GRIS) for c in participacion.columns]
ax = participacion.plot.barh(stacked=True, figsize=(11, 5), color=colores, edgecolor="white", linewidth=1)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set_title("P2. Participación de cada aerolínea en las cancelaciones de la ruta")
ax.set_xlabel("Parte de las cancelaciones de la ruta")
ax.set_ylabel(None)
ax.legend(title=None, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

**Respuesta P2** (completar con la salida al correrlo en la VM): qué rutas encabezan y con qué tasa, y si las cancelaciones de cada una se concentran en una o dos aerolíneas.

## 4. P3: atraso por hora y día en los 10 aeropuertos con más movimiento

Cómo se mide:

- **Movimiento**: salidas más llegadas realizadas en el año.
- **Atraso promedio de salida**: promedio de `atraso_partida_min` de los vuelos medidos que salen de esos aeropuertos. Los adelantos restan.
- **Hora y día en hora local** del aeropuerto (`hora_local` y `dia_semana_local`, del notebook 1). La primera celda muestra el huso de los 10: si todos son UTC-3, la hora local coincide con la de Brasília, que es la que informa ANAC.
- Las combinaciones con menos de 30 vuelos quedan afuera: el promedio de tan pocos vuelos no es representativo.

In [ ]:
p3_top = spark.sql(f"""
SELECT m.icao_aeropuerto, a.nombre_aeropuerto, a.ciudad, a.uf, a.huso_utc, m.movimientos
FROM (
    SELECT icao_aeropuerto, COUNT(*) AS movimientos
    FROM (SELECT icao_origem AS icao_aeropuerto FROM {DB}.fact_vuelo WHERE realizado
          UNION ALL
          SELECT icao_destino AS icao_aeropuerto FROM {DB}.fact_vuelo WHERE realizado) t
    GROUP BY icao_aeropuerto
) m
JOIN {DB}.dim_aeropuerto a ON m.icao_aeropuerto = a.icao_aeropuerto
ORDER BY m.movimientos DESC, m.icao_aeropuerto
LIMIT 10
""")
p3_top.createOrReplaceTempView("p3_top")
p3_top.show(truncate=40)
print("Husos de los 10 aeropuertos:", sorted({fila["huso_utc"] for fila in p3_top.collect()}))

In [ ]:
p3_dia_hora = spark.sql(f"""
SELECT f.dia_semana_local, f.hora_local, COUNT(*) AS vuelos,
       ROUND(AVG(f.atraso_partida_min), 1) AS atraso_medio
FROM {DB}.fact_vuelo f
JOIN p3_top t ON f.icao_origem = t.icao_aeropuerto
WHERE f.atraso_partida_min IS NOT NULL
GROUP BY f.dia_semana_local, f.hora_local
HAVING COUNT(*) >= 30
""")
p3_aeropuerto_hora = spark.sql(f"""
SELECT t.icao_aeropuerto, t.ciudad, f.hora_local, COUNT(*) AS vuelos,
       ROUND(AVG(f.atraso_partida_min), 1) AS atraso_medio
FROM {DB}.fact_vuelo f
JOIN p3_top t ON f.icao_origem = t.icao_aeropuerto
WHERE f.atraso_partida_min IS NOT NULL
GROUP BY t.icao_aeropuerto, t.ciudad, f.hora_local
HAVING COUNT(*) >= 30
""")
p3_dia_hora.orderBy("atraso_medio", ascending=False).show(5)

In [ ]:
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
dia_hora = p3_dia_hora.toPandas()
dia_hora["dia"] = dia_hora["dia_semana_local"].map(lambda d: DIAS[d - 1])
mapa = (dia_hora.pivot(index="dia", columns="hora_local", values="atraso_medio")
        .reindex(index=DIAS, columns=range(24)))

# Gráfico 1: mapa de calor de día por hora, con los 10 aeropuertos juntos
fig, ax = plt.subplots(figsize=(15, 4.5))
sns.heatmap(mapa, cmap="vlag", center=0, annot=True, fmt=".0f", annot_kws={"size": 7},
            cbar_kws={"label": "Atraso medio de salida (min)"}, ax=ax)
ax.set_title("P3. Atraso medio de salida por día y hora local, en los 10 aeropuertos con más movimiento")
ax.set_xlabel("Hora local de salida prevista")
ax.set_ylabel(None)
plt.tight_layout()
plt.show()

# Gráfico 2: atraso medio por hora, un panel por aeropuerto (diez líneas en un mismo eje no se leen)
aeropuerto_hora = p3_aeropuerto_hora.toPandas()
fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharex=True, sharey=True)
for ax, fila in zip(axes.flat, p3_top.toPandas().fillna("").itertuples()):
    datos = aeropuerto_hora[aeropuerto_hora["icao_aeropuerto"] == fila.icao_aeropuerto].sort_values("hora_local")
    ax.plot(datos["hora_local"], datos["atraso_medio"], marker="o", markersize=3)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_title(f"{fila.icao_aeropuerto} {fila.ciudad}", fontsize=9)
    ax.set_xticks(range(0, 24, 3))
for ax in axes[1]:
    ax.set_xlabel("Hora local de salida")
for ax in axes[:, 0]:
    ax.set_ylabel("Atraso medio (min)")
fig.suptitle("P3. Atraso medio de salida por hora local, un panel por aeropuerto (ordenados por movimiento)")
plt.tight_layout()
plt.show()

**Respuesta P3** (completar con la salida al correrlo en la VM): en qué horas y días sube el atraso, y qué aeropuertos se apartan del resto.

## 5. P4 y P5: tablas para Superset

Estas dos preguntas se responden en el dashboard de Superset. Acá se arman sus tablas con el patrón que se vio en clase para la zona analítica:

1. Una base aparte, `resumen_vuelos_brasil`, que agrupa las respuestas. Es la que se conecta desde Superset.
2. Cada tabla es **EXTERNAL**, en parquet, con `LOCATION` en `/anl/vuelos_brasil/v1/<tabla>`. No se usan vistas: la consigna pide guardar los resultados como archivos en la zona de analítica.
3. Se carga con `INSERT OVERWRITE` a partir de la consulta sobre el modelo. Volver a correr la celda reemplaza los datos, que es la forma de actualizar en Hive.

El DDL se arma con el esquema del resultado de la consulta: así los tipos y el orden de las columnas siempre coinciden, porque Hive inserta por posición y no por nombre. Además de las tasas, cada tabla guarda los conteos con que se calculan, para que Superset pueda recalcular una tasa cuando se filtra (suma de puntuales sobre suma de medidos) en lugar de promediar porcentajes.

Si el `INSERT` da *Permission denied*, la carpeta de la tabla la creó Hive con otro usuario: repetir `docker exec namenode hdfs dfs -chmod -R 777 /anl` y volver a correr la celda.

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_ANL}")
ddls = {}

def guardar_tabla(nombre, df):
    # Tabla externa en /anl con el esquema de la consulta, cargada con INSERT OVERWRITE
    columnas = ",\n  ".join(f"`{c.name}` {c.dataType.simpleString()}" for c in df.schema.fields)
    ddl = (f"CREATE EXTERNAL TABLE {DB_ANL}.{nombre} (\n  {columnas}\n)\n"
           f"STORED AS PARQUET\nLOCATION '{ANL}/{nombre}'")
    spark.sql(f"DROP TABLE IF EXISTS {DB_ANL}.{nombre}")   # externa: el DROP no borra los archivos
    spark.sql(ddl)
    df.createOrReplaceTempView(f"tmp_{nombre}")
    spark.sql(f"INSERT OVERWRITE TABLE {DB_ANL}.{nombre} SELECT * FROM tmp_{nombre}")
    en_hive, en_consulta = spark.table(f"{DB_ANL}.{nombre}").count(), df.count()
    print(f"[{'OK' if en_hive == en_consulta else 'DIFERENTE'}] {DB_ANL}.{nombre}: "
          f"Hive {en_hive:,} / consulta {en_consulta:,}")
    ddls[nombre] = ddl + ";"

### P4: frecuencias semanales por país

Cómo se mide:

- **Frecuencia semanal**: vuelos realizados del trimestre sobre semanas del trimestre (días del trimestre sobre 7).
- Se cuenta la **salida desde Brasil** hacia el país. Una frecuencia es una ida y vuelta, así que 7 frecuencias semanales equivalen a un vuelo diario; contar los dos sentidos duplicaría el número sin agregar información.
- **País de destino**: el del aeropuerto de llegada, según `dim_aeropuerto` y `dim_pais`.
- **Participación brasileña**: vuelos de empresas brasileñas sobre vuelos hacia ese país, en el trimestre. Las empresas de nacionalidad DESCONOCIDA cuentan en el total.

| Tabla | Grano | Para qué |
|---|---|---|
| `p4_frecuencias` | País, trimestre y nacionalidad de la empresa | Brasileñas contra extranjeras, y filtros por trimestre, continente y país |
| `p4_cambio_pais` | País | Cambio de frecuencia entre el primer y el cuarto trimestre (Uruguay destacado) y tabla de detalle |

In [ ]:
# Salidas desde Brasil cuyo país de destino no se pudo identificar: quedan afuera
spark.sql(f"""
SELECT COUNT(*) AS salidas_sin_pais
FROM {DB}.fact_vuelo f
JOIN {DB}.dim_aeropuerto o ON f.icao_origem = o.icao_aeropuerto
JOIN {DB}.dim_aeropuerto a ON f.icao_destino = a.icao_aeropuerto
WHERE f.realizado AND o.iso_pais = 'BR' AND a.iso_pais IS NULL
""").show()

p4_frecuencias = spark.sql(f"""
SELECT p.nombre_pais AS pais, p.iso_pais, p.continente, (p.iso_pais = 'UY') AS es_uruguay,
       d.trimestre, e.nacionalidad,
       COUNT(*) AS vuelos_realizados,
       ROUND(COUNT(*) / s.semanas, 2) AS frecuencia_semanal
FROM {DB}.fact_vuelo f
JOIN {DB}.dim_aeropuerto o ON f.icao_origem = o.icao_aeropuerto
JOIN {DB}.dim_aeropuerto a ON f.icao_destino = a.icao_aeropuerto
JOIN {DB}.dim_pais p ON a.iso_pais = p.iso_pais
JOIN {DB}.dim_empresa e ON f.icao_empresa = e.icao_empresa
JOIN {DB}.dim_fecha d ON f.fecha = d.fecha
JOIN (SELECT trimestre, COUNT(*) / 7 AS semanas FROM {DB}.dim_fecha GROUP BY trimestre) s
  ON d.trimestre = s.trimestre
WHERE f.realizado AND o.iso_pais = 'BR' AND a.iso_pais <> 'BR'
GROUP BY p.nombre_pais, p.iso_pais, p.continente, d.trimestre, e.nacionalidad, s.semanas
""")
guardar_tabla("p4_frecuencias", p4_frecuencias)

In [ ]:
# El cambio por país sale de la tabla de frecuencias, que ya está en la zona de analítica
p4_cambio_pais = spark.sql(f"""
SELECT pais, iso_pais, continente, es_uruguay,
       ROUND(frecuencia_t1, 2) AS frecuencia_t1,
       ROUND(frecuencia_t4, 2) AS frecuencia_t4,
       ROUND(frecuencia_t4 - frecuencia_t1, 2) AS diferencia,
       ROUND(brasilenas_t1 / NULLIF(vuelos_t1, 0), 4) AS participacion_br_t1,
       ROUND(brasilenas_t4 / NULLIF(vuelos_t4, 0), 4) AS participacion_br_t4
FROM (
    SELECT pais, iso_pais, continente, es_uruguay,
           SUM(IF(trimestre = 1, frecuencia_semanal, 0)) AS frecuencia_t1,
           SUM(IF(trimestre = 4, frecuencia_semanal, 0)) AS frecuencia_t4,
           SUM(IF(trimestre = 1, vuelos_realizados, 0)) AS vuelos_t1,
           SUM(IF(trimestre = 4, vuelos_realizados, 0)) AS vuelos_t4,
           SUM(IF(trimestre = 1 AND nacionalidad = 'BRASILEÑA', vuelos_realizados, 0)) AS brasilenas_t1,
           SUM(IF(trimestre = 4 AND nacionalidad = 'BRASILEÑA', vuelos_realizados, 0)) AS brasilenas_t4
    FROM {DB_ANL}.p4_frecuencias
    GROUP BY pais, iso_pais, continente, es_uruguay
) t
""")
guardar_tabla("p4_cambio_pais", p4_cambio_pais)

cambio = f"SELECT * FROM {DB_ANL}.p4_cambio_pais"
spark.sql(f"{cambio} ORDER BY diferencia LIMIT 10").show(truncate=25)        # los que más perdieron
spark.sql(f"{cambio} ORDER BY diferencia DESC LIMIT 10").show(truncate=25)   # los que más ganaron
spark.sql(f"{cambio} WHERE es_uruguay").show(truncate=25)

### P5: puntualidad y cancelación por tamaño de aeropuerto

Cómo se mide:

- **Tamaño**: la clasificación de OurAirports (`large_airport`, `medium_airport`, `small_airport`) que el notebook 1 guardó en `dim_aeropuerto.tamano` como GRANDE, MEDIANO y CHICO.
- El aeropuerto de cada vuelo es el de **origen**, porque la puntualidad se mide en la salida.
- **Solo aeropuertos de Brasil.** Los extranjeros solo tienen vuelos internacionales y casi todos son grandes: mezclarlos cambiaría la comparación.
- Quedan afuera los vuelos que salen de aeropuertos sin esa clasificación (OTRO o DESCONOCIDO). La primera celda los cuenta.
- Puntualidad y cancelación con las definiciones de la sección 1, mes a mes.

`kpi_resumen` es el resumen mensual de todos los vuelos programados, para la fila de indicadores del dashboard.

In [ ]:
spark.sql(f"""
SELECT a.tamano, COUNT(*) AS programados
FROM {DB}.fact_vuelo f
JOIN {DB}.dim_aeropuerto a ON f.icao_origem = a.icao_aeropuerto
WHERE a.iso_pais = 'BR' AND f.tiene_horario_previsto AND f.informado
GROUP BY a.tamano
ORDER BY programados DESC
""").show()

p5_tamano_mes = spark.sql(f"""
SELECT a.tamano, f.anio_mes, d.trimestre,
       COUNT(*)                      AS programados,
       SUM(CAST(f.realizado AS INT)) AS realizados,
       SUM(CAST(f.cancelado AS INT)) AS cancelados,
       COUNT(f.puntual)              AS vuelos_medidos,
       SUM(CAST(f.puntual AS INT))   AS vuelos_puntuales,
       ROUND(SUM(CAST(f.puntual AS INT)) / NULLIF(COUNT(f.puntual), 0), 4) AS tasa_puntualidad,
       ROUND(SUM(CAST(f.cancelado AS INT)) / COUNT(*), 4)                 AS tasa_cancelacion
FROM {DB}.fact_vuelo f
JOIN {DB}.dim_aeropuerto a ON f.icao_origem = a.icao_aeropuerto
JOIN {DB}.dim_fecha d ON f.fecha = d.fecha
WHERE a.iso_pais = 'BR' AND a.tamano IN ('GRANDE', 'MEDIANO', 'CHICO')
  AND f.tiene_horario_previsto AND f.informado
GROUP BY a.tamano, f.anio_mes, d.trimestre
""")
guardar_tabla("p5_tamano_mes", p5_tamano_mes)

# Resultado del año, recalculado con los conteos de la tabla
spark.sql(f"""
SELECT tamano, SUM(programados) AS programados,
       ROUND(SUM(vuelos_puntuales) / SUM(vuelos_medidos), 4) AS tasa_puntualidad,
       ROUND(SUM(cancelados) / SUM(programados), 4)          AS tasa_cancelacion
FROM {DB_ANL}.p5_tamano_mes
GROUP BY tamano
ORDER BY programados DESC
""").show()

In [ ]:
kpi_resumen = spark.sql(f"""
SELECT f.anio_mes, d.trimestre,
       COUNT(*)                      AS vuelos_programados,
       SUM(CAST(f.realizado AS INT)) AS vuelos_realizados,
       SUM(CAST(f.cancelado AS INT)) AS vuelos_cancelados,
       COUNT(f.puntual)              AS vuelos_medidos,
       SUM(CAST(f.puntual AS INT))   AS vuelos_puntuales,
       ROUND(SUM(CAST(f.puntual AS INT)) / NULLIF(COUNT(f.puntual), 0), 4) AS tasa_puntualidad,
       ROUND(SUM(CAST(f.cancelado AS INT)) / COUNT(*), 4)                 AS tasa_cancelacion,
       p.paises_destino
FROM {DB}.fact_vuelo f
JOIN {DB}.dim_fecha d ON f.fecha = d.fecha
LEFT JOIN (
    SELECT v.anio_mes, COUNT(DISTINCT a.iso_pais) AS paises_destino
    FROM {DB}.fact_vuelo v
    JOIN {DB}.dim_aeropuerto o ON v.icao_origem = o.icao_aeropuerto
    JOIN {DB}.dim_aeropuerto a ON v.icao_destino = a.icao_aeropuerto
    WHERE v.realizado AND o.iso_pais = 'BR' AND a.iso_pais <> 'BR'
    GROUP BY v.anio_mes
) p ON f.anio_mes = p.anio_mes
WHERE f.tiene_horario_previsto AND f.informado
GROUP BY f.anio_mes, d.trimestre, p.paises_destino
""")
guardar_tabla("kpi_resumen", kpi_resumen)
spark.sql(f"SELECT * FROM {DB_ANL}.kpi_resumen ORDER BY anio_mes").show()

## 6. Validación de las tablas de analítica

El mismo control que en el notebook 1: las tablas existen, son externas y apuntan a `/anl`. El DDL impreso va al repo como `sql/ddl_analitica.sql`. Para el informe, sumar la salida de `docker exec namenode hdfs dfs -ls -R /anl/vuelos_brasil/v1`.

In [ ]:
spark.sql(f"SHOW TABLES IN {DB_ANL}").show()
for nombre in ddls:
    (spark.sql(f"DESCRIBE FORMATTED {DB_ANL}.{nombre}")
     .filter("col_name IN ('Type', 'Location')")
     .show(truncate=False))
print("\n\n".join(ddls.values()))   # este texto va al repo como sql/ddl_analitica.sql

## Lo que sigue: Superset

En Superset, un dataset por tabla (Datasets, + Dataset, base Apache Hive, esquema `resumen_vuelos_brasil`):

| Tabla | Bloque del dashboard |
|---|---|
| `kpi_resumen` | Fila de indicadores: vuelos programados, puntualidad, cancelación y países de destino |
| `p4_cambio_pais` | P4: cambio de frecuencia semanal por país entre el primer y el cuarto trimestre, con Uruguay destacado, y tabla de detalle |
| `p4_frecuencias` | P4: brasileñas contra extranjeras, por país y trimestre |
| `p5_tamano_mes` | P5: puntualidad y cancelación mes a mes por tamaño de aeropuerto |

Para que una tasa siga siendo correcta al filtrar, la métrica de Superset se arma con los conteos, por ejemplo `SUM(vuelos_puntuales) / SUM(vuelos_medidos)`.